# 02 · Cleaning

All the logic lives in `src/ufc/` so it's reusable later (simulator, API, website):
- `ingest.py` parses the Greco1899 snapshot (see 01b).
- `cleaning.py` holds the shared rules (weight classes, method groups, the long table) and the original 2024-file pipeline.

What the pipeline does:
1. Parses `"9 of 37"`‑style strings, clock times, heights (ft/in → cm), weights (lbs → kg) and **birthdates**.
2. Deduplicates fights by URL, attaches **dates**, and resolves fighter identities (spelling variants, shared names).
3. Collapses 100+ weight-class labels ("UFC Women's Flyweight Title Bout", "Road to UFC 3 Flyweight Tournament TitleBout", …) into 15 divisions.
4. Groups methods into `KO/TKO`, `SUB`, `DEC`, `Other` (DQ, overturned, could not continue).
5. Computes **exact age at fight** from DOB, and real fight length from each bout's round format.
6. Sorts chronologically and assigns `fight_id` (time order, so same-night tournament bouts are ordered too).
7. Builds a **long table** (two rows per fight, one per fighter, with their own and their opponent's stats), plus a **round table** and a **judges table**.

In [1]:
import os, sys
from pathlib import Path
# Run from the project root no matter where Jupyter was started.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)

In [2]:
from ufc import ingest
fights, ff, rounds, judges, report = ingest.run()
print(fights.shape, ff.shape, rounds.shape, judges.shape)
fights.head()

(8947, 80) (17894, 62) (41996, 27) (12100, 6)


,fight_id,fight_url_id,date,event,location,bout_order,r_fighter,b_fighter,winner,red_win,corners_reliable,weight_class,weight_class_raw,gender,is_title_bout,method,method_group,method_detail,finish_round,total_rounds,time_sec,time_format,fight_seconds,referee,r_kd,r_sig_str,r_sig_str_att,r_str,r_str_att,r_td,...,b_str_att,b_td,b_td_att,b_sub_att,b_rev,b_ctrl_sec,b_head,b_head_att,b_body,b_body_att,b_leg,b_leg_att,b_distance,b_distance_att,b_clinch,b_clinch_att,b_ground,b_ground_att,r_height,r_reach,r_weight,r_stance,r_age_at_fight,r_fighter_url_id,b_height,b_reach,b_weight,b_stance,b_age_at_fight,b_fighter_url_id
0,0,4acab67848e78327,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",0,Scott Morris,Sean Daugherty,Red,1.0,False,Open Weight,Open Weight Bout,Men,0,Submission,SUB,Guillotine Choke From Mount,1,1,20.0,No Time Limit,20.0,John McCarthy,0.0,1.0,1.0,2.0,2.0,1.0,...,5.0,0.0,0.0,0.0,0.0,NaN,0.0,2.0,0.0,0.0,0.0,2.0,0.0,3.0,0.0,1.0,0.0,0.0,177.80,NaN,95.25,Orthodox,NaN,be9d259be012e8a4,182.88,NaN,79.38,NaN,18.3,a683f9ddb70aa4bd
1,1,4b9ae533ccb3fcdf,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",1,Patrick Smith,Ray Wizard,Red,1.0,False,Open Weight,Open Weight Bout,Men,0,Submission,SUB,Guillotine Choke Standing,1,1,58.0,No Time Limit,58.0,John McCarthy,0.0,1.0,1.0,1.0,1.0,0.0,...,2.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,187.96,NaN,102.06,Orthodox,30.5,46c8ec317aff28ac,NaN,NaN,NaN,NaN,NaN,ea0ad155451ed1f5
2,2,c53325be8ce33b8b,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",2,Jason DeLucia,Scott Baker,Red,1.0,False,Open Weight,Open Weight Bout,Men,0,Submission,SUB,Triangle Choke From Mount,1,1,401.0,No Time Limit,401.0,John McCarthy,0.0,3.0,5.0,20.0,25.0,0.0,...,23.0,1.0,1.0,0.0,2.0,NaN,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,180.34,NaN,86.18,Southpaw,24.6,6ceff86fae4f6b3b,182.88,NaN,95.25,NaN,NaN,7d5b12de1625984e
3,3,fcd8e8ec45eb6b9f,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",3,Royce Gracie,Minoki Ichihara,Red,1.0,False,Open Weight,Open Weight Bout,Men,0,Submission,SUB,Armbar On Ground,1,1,308.0,No Time Limit,308.0,John McCarthy,0.0,2.0,4.0,110.0,114.0,1.0,...,16.0,0.0,0.0,0.0,0.0,NaN,3.0,6.0,0.0,0.0,0.0,1.0,0.0,2.0,0.0,0.0,3.0,5.0,185.42,NaN,79.38,Southpaw,27.2,429e7d3725852ce9,170.18,NaN,80.74,Orthodox,NaN,4565d435005319c0
4,4,8ea72cf82fac07f6,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",4,Patrick Smith,Scott Morris,Red,1.0,False,Open Weight,Open Weight Bout,Men,0,KO/TKO,KO/TKO,Elbows to Head From Mount,1,1,30.0,No Time Limit,30.0,John McCarthy,0.0,13.0,17.0,19.0,25.0,0.0,...,0.0,0.0,1.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,187.96,NaN,102.06,Orthodox,30.5,46c8ec317aff28ac,177.80,NaN,95.25,Orthodox,NaN,be9d259be012e8a4


## Before → after: weight classes

In [3]:
(fights.groupby("weight_class").weight_class_raw.agg(["nunique", "size"])
       .rename(columns={"nunique": "raw labels", "size": "fights"}).sort_values("fights", ascending=False))

,raw labels,fights
weight_class,,
Lightweight,16,1489
Welterweight,17,1422
Middleweight,16,1177
Featherweight,10,892
Bantamweight,9,810
Heavyweight,12,795
Light Heavyweight,8,778
Flyweight,6,432
Women's Strawweight,4,382


## Methods, results, and age at fight

In [4]:
display(fights.method_group.value_counts())
display(fights.winner.value_counts())
fights[["r_age_at_fight", "b_age_at_fight"]].describe().round(1)

method_group
DEC       4141
KO/TKO    2956
SUB       1734
Other      116
Name: count, dtype: int64

winner
Red     5622
Blue    3170
NC        89
Draw      66
Name: count, dtype: int64

,r_age_at_fight,b_age_at_fight
count,8870.0,8749.0
mean,30.7,30.3
std,4.3,4.0
min,18.2,18.3
25%,27.6,27.5
50%,30.6,30.1
75%,33.5,32.9
max,47.2,51.9


Age now comes from real birthdates. Results include **no-contests** (`NC`): they count toward experience but are never prediction targets, and they don't break win/loss streaks.

## Sanity checks on physical measurements (cm / kg)

In [5]:
fights[["r_height","r_reach","r_weight","b_height","b_reach","b_weight"]].describe().round(1)

,r_height,r_reach,r_weight,b_height,b_reach,b_weight
count,8887.0,8473.0,8938.0,8862.0,7981.0,8918.0
mean,178.5,183.1,76.0,178.4,182.7,75.8
std,9.1,11.0,16.5,9.0,10.9,16.7
min,152.4,147.3,52.2,152.4,147.3,52.2
25%,172.7,175.3,61.2,172.7,175.3,61.2
50%,177.8,182.9,70.3,177.8,182.9,70.3
75%,185.4,190.5,83.9,185.4,190.5,83.9
max,210.8,213.4,156.5,210.8,213.4,349.3


The ranges look physically plausible, so no unit errors. `weight` is the fighter's *listed* weight, not the weight on fight night, so we won't use it as a feature. Weight class covers it.

## The long fighter‑fight table
Each fight appears twice: once from the red fighter's point of view, once from the blue fighter's.
`opp_*` columns hold the opponent's numbers for that fight, now including strike targets and positions.

In [6]:
ff[ff.fighter == "Israel Adesanya"][["date","fighter","opponent","corner","result","method_group",
                                       "sig_str","opp_sig_str","head","leg","td","height","reach","age_at_fight"]].tail(6)

,date,fighter,opponent,corner,result,method_group,sig_str,opp_sig_str,head,leg,td,height,reach,age_at_fight
13761,2022-11-12,Israel Adesanya,Alex Pereira,Red,L,KO/TKO,86.0,91.0,41.0,24.0,1.0,193.04,203.2,33.3
14136,2023-04-08,Israel Adesanya,Alex Pereira,Blue,W,KO/TKO,41.0,49.0,18.0,9.0,0.0,193.04,203.2,33.7
14655,2023-09-09,Israel Adesanya,Sean Strickland,Red,L,DEC,94.0,137.0,22.0,34.0,0.0,193.04,203.2,34.1
15580,2024-08-17,Israel Adesanya,Dricus Du Plessis,Blue,L,SUB,97.0,90.0,63.0,8.0,0.0,193.04,203.2,35.1
16019,2025-02-01,Israel Adesanya,Nassourdine Imavov,Red,L,KO/TKO,26.0,15.0,9.0,11.0,0.0,193.04,203.2,35.5
17215,2026-03-28,Israel Adesanya,Joe Pyfer,Red,L,KO/TKO,42.0,36.0,22.0,12.0,0.0,193.04,203.2,36.7


## The round table
One row per fighter per round. This is what will let the simulator model cardio and round-by-round judging.

In [7]:
rounds[rounds.fighter.isin(["Israel Adesanya"])].tail(6)

,fight_url_id,corner,round,kd,sig_str,sig_str_att,str,str_att,td,td_att,sub_att,rev,ctrl_sec,head,head_att,body,body_att,leg,leg_att,distance,distance_att,clinch,clinch_att,ground,ground_att,fight_id,fighter
36486,6c44af21414e4143,Blue,3,0.0,40.0,83.0,40.0,83.0,0.0,0.0,0.0,0.0,0.0,25.0,66.0,12.0,14.0,3.0,3.0,40.0,83.0,0.0,0.0,0.0,0.0,7790,Israel Adesanya
36488,6c44af21414e4143,Blue,4,0.0,24.0,53.0,24.0,53.0,0.0,0.0,0.0,0.0,7.0,16.0,43.0,8.0,9.0,0.0,1.0,23.0,52.0,0.0,0.0,1.0,1.0,7790,Israel Adesanya
37607,85e79748b75eb30e,Red,1,0.0,20.0,44.0,26.0,50.0,0.0,1.0,0.0,0.0,0.0,8.0,27.0,3.0,5.0,9.0,12.0,20.0,44.0,0.0,0.0,0.0,0.0,8009,Israel Adesanya
37609,85e79748b75eb30e,Red,2,0.0,6.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,3.0,3.0,2.0,2.0,6.0,6.0,0.0,0.0,0.0,0.0,8009,Israel Adesanya
40491,85e94a6c071fd9fa,Red,1,0.0,24.0,42.0,37.0,56.0,0.0,0.0,0.0,0.0,0.0,13.0,30.0,2.0,2.0,9.0,10.0,24.0,42.0,0.0,0.0,0.0,0.0,8607,Israel Adesanya
40493,85e94a6c071fd9fa,Red,2,0.0,18.0,33.0,21.0,36.0,0.0,2.0,0.0,0.0,0.0,9.0,24.0,6.0,6.0,3.0,3.0,18.0,32.0,0.0,1.0,0.0,0.0,8607,Israel Adesanya


In [8]:
assert fights.fight_id.is_unique
assert fights.date.is_monotonic_increasing
assert len(ff) == 2 * len(fights)
assert set(ff.result.unique()) <= {"W", "L", "D", "N"}
print("all checks passed")

all checks passed


Saved to `data/processed/`: `fights`, `fighter_fights`, `fight_rounds`, `judge_scores` (parquet). → `03_eda.ipynb`